# Étape 3 — Récompenses et gradients sur Colab T4

**Une image, premier prompt dev, seed 0.** SD1.5, 512×512, DDIM 50, eta=0, CFG 7.5.
Ce notebook vérifie les récompenses ; il n'applique aucune guidance de récompense.
La baseline reste dans `baseline.py`, les nouvelles fonctions dans `rewards.py`.

1. Choisir un runtime **GPU T4**, importer **`diffusion-reward-guidance.zip`** dans la première cellule.
2. Exécuter les deux premières cellules de code. Si l'installation affiche STOP,
   **redémarrer la session**, puis reprendre depuis la première cellule.
3. Ensuite, exécuter toutes les cellules dans l'ordre. Une assertion échouée est un échec à analyser.
4. Télécharger l'archive des résultats à la fin, avant de fermer le runtime.

**État :** tests CPU exécutés ; le diagnostic T4 transmis par l'utilisateur a motivé une grille
ImageReward plus fine. Cette version corrigée reste à exécuter sur GPU et ne contient aucun résultat
prérempli. Voir `validation/` pour distinguer les mesures déjà obtenues des tests à refaire.
Le checkpoint ImageReward représente environ 1,8 Go à télécharger en plus de SD1.5.

In [ ]:
from pathlib import Path
import os
import sys
import zipfile

candidates = [Path.cwd(), Path.cwd().parent, Path('/content/diffusion-reward-guidance')]
PROJECT_DIR = next((p for p in candidates if (p / 'rewards.py').is_file()), None)
if PROJECT_DIR is None:
    from google.colab import files
    print('Importer diffusion-reward-guidance.zip (la nouvelle archive complète).')
    uploaded = files.upload()
    archive_name = next(name for name in uploaded if name.endswith('.zip'))
    destination = Path('/content').resolve()
    with zipfile.ZipFile(archive_name) as archive:
        for entry in archive.infolist():
            (destination / entry.filename).resolve().relative_to(destination)
        archive.extractall(destination)
    PROJECT_DIR = destination / 'diffusion-reward-guidance'
assert (PROJECT_DIR / 'rewards.py').is_file(), 'Utiliser la nouvelle archive rewards.'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print('Projet :', PROJECT_DIR)

## 1. Installation figée

L'environnement BASE est conservé. `requirements-rewards.txt` ajoute les dépendances de l'étape 3.
Le code officiel ImageReward est figé à un commit compatible avec notre version de Transformers.
Son paquet importe aussi OpenAI CLIP : cette dépendance est installée, **aucune métrique CLIP n'est calculée**.
Les commits, poids et tokenizer sont figés et enregistrés. Les dépendances transitives réellement
résolues sont conservées par `pip freeze` avec les résultats.

Les sorties de pip sont visibles et sauvegardées dans `installation_logs/`.
Fairscale et les deux dépôts officiels sont des paquets Python construits à l'installation.
Ne pas reprendre les calculs avant le redémarrage demandé.

In [ ]:
import importlib.metadata
import json
import subprocess

assert (3, 10) <= sys.version_info[:2] <= (3, 13), 'Utiliser Python 3.10 à 3.13.'
expected = {}
for filename in ('requirements.txt', 'requirements-rewards.txt'):
    for line in (PROJECT_DIR / filename).read_text(encoding='utf-8').splitlines():
        if '==' in line and not line.lstrip().startswith('#'):
            package, version = line.strip().split('==')
            expected[package] = version
torch_expected = {'torch': '2.8.0+cu126', 'torchvision': '0.23.0+cu126', 'torchaudio': '2.8.0+cu126'}
expected.update(torch_expected)
source_urls = {
    'clip': 'https://github.com/openai/CLIP/archive/d05afc436d78f1c48dc0dbf8e5980a9d471f35f6.zip',
    'image-reward': 'https://github.com/zai-org/ImageReward/archive/78f8051059b34fef8e0032ccb40561c23a2ddbbb.zip',
}
installed = {}
for package in expected:
    try:
        installed[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed[package] = None
sources_match = {}
for package, url in source_urls.items():
    try:
        metadata = importlib.metadata.distribution(package).read_text('direct_url.json')
        sources_match[package] = json.loads(metadata or '{}').get('url') == url
    except importlib.metadata.PackageNotFoundError:
        sources_match[package] = False
needs_install = any(installed[p] != v for p, v in expected.items()) or not all(sources_match.values())
RESTART_REQUIRED = globals().get('RESTART_REQUIRED', False) or needs_install

def run_visible(command, log_name):
    log_dir = PROJECT_DIR / 'installation_logs'
    log_dir.mkdir(exist_ok=True)
    with (log_dir / log_name).open('w', encoding='utf-8') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
            return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Échec {return_code} ; consulter les lignes ci-dessus et {log_dir / log_name}.')

if needs_install:
    if any(installed[p] != v for p, v in torch_expected.items()):
        run_visible([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off',
                     'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
                     '--index-url', 'https://download.pytorch.org/whl/cu126'], 'rewards-pytorch.log')
    run_visible([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off',
                 '-r', 'requirements-rewards.txt'], 'rewards-dependencies.log')
    for package, url in source_urls.items():
        if not sources_match[package]:
            # Les dépendances sont installées au-dessus : ne pas résoudre CLIP depuis main.
            run_visible([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation',
                         '--progress-bar', 'off', f'{package} @ {url}'], f'rewards-{package}.log')
if RESTART_REQUIRED:
    print('STOP : redémarrer la session, puis reprendre à la première cellule.')
else:
    print('Versions et commits conformes ; vous pouvez exécuter la suite.')

In [ ]:
assert not RESTART_REQUIRED, 'Redémarrer réellement la session avant les imports.'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import gc
import hashlib
from datetime import datetime, timezone
import torch
from IPython.display import display
from PIL import Image

from baseline import configure_torch, load_pipeline, decode_latents, measure_call
from validate_baseline import select_dev_prompts, save_environment, validate_prompt, compare_tensors
from rewards import (
    luminance_reward, symmetry_reward, spatial_mask_reward, make_disk_mask,
    decode_latents_differentiable, load_imagereward, image_reward,
    check_latent_gradient, compare_imagereward_api,
    IMAGEREWARD_CODE_REVISION, IMAGEREWARD_MODEL_REVISION, IMAGEREWARD_WEIGHTS_SHA256, BERT_REVISION,
    IMAGEREWARD_FD_STEPS,
)

configure_torch()
assert torch.cuda.is_available(), 'Activer un runtime GPU dans Colab.'
assert all(importlib.metadata.version(p) == v for p, v in expected.items())
assert all(sources_match.values()), 'La version PyPI seule ne suffit pas : contrôler le commit officiel.'
print('GPU :', torch.cuda.get_device_name(0), '| PyTorch :', torch.__version__)
print('Déterminisme strict :', torch.are_deterministic_algorithms_enabled())
run_visible([sys.executable, '-m', 'unittest', '-v', 'test_baseline', 'test_rewards'], 'rewards-tests-cpu.log')

## 2. Une image BASE et sa validation de régression

Le premier prompt dev et la seed 0 restent fixes. Les deux chemins BASE génèrent la même image
pour contrôler que l'ajout des dépendances n'a pas changé la parité avec Diffusers.
Le coût de ce contrôle inclut les traces ; ce n'est pas un benchmark de débit.
Les latents finaux sont sauvegardés puis réutilisés par les quatre récompenses.

In [ ]:
PROMPT = select_dev_prompts(1)[0]
SEED = 0
TARGET_LUMINANCE = 0.7  # Choix provisoire explicite, commun à tous les contrôles.
MASK_RADIUS_FRACTION = 0.25
RUN_DIR = PROJECT_DIR / 'runs' / datetime.now(timezone.utc).strftime('rewards_%Y%m%d_%H%M%S_%f_UTC')
pipe = load_pipeline()
save_environment(pipe, RUN_DIR)

reward_environment = {
    'prompt': PROMPT, 'seed': SEED, 'luminance_target': TARGET_LUMINANCE,
    'mask': {'radius_fraction': MASK_RADIUS_FRACTION, 'bright': 1.0, 'dark': 0.0},
    'image_reward_code_revision': IMAGEREWARD_CODE_REVISION,
    'image_reward_weights_revision': IMAGEREWARD_MODEL_REVISION,
    'image_reward_weights_sha256': IMAGEREWARD_WEIGHTS_SHA256,
    'bert_revision': BERT_REVISION, 'official_source_urls': source_urls,
    'versions': {p: importlib.metadata.version(p) for p in (*expected, *source_urls)},
    'gradient_dtype': 'float32', 'vae_gradient_checkpointing': True,
    'unet_and_text_encoder_during_gradient_tests': 'CPU', 'bicubic_resize_device': 'CPU',
    'finite_difference_steps': {'analytic': [0.01, 0.003, 0.001], 'imagereward': list(IMAGEREWARD_FD_STEPS)},
    'min_consecutive_steps': {'analytic': 1, 'imagereward': 2},
    'finite_difference_mode': 'enable_grad_with_leaf_requires_grad',
    'fd_atol': 1e-5, 'fd_rtol': 0.10,
    'source_sha256': {name: hashlib.sha256((PROJECT_DIR / name).read_bytes()).hexdigest()
                      for name in ('rewards.py', 'test_rewards.py', 'requirements-rewards.txt',
                                   'notebooks/02_rewards_t4.ipynb')},
}
(RUN_DIR / 'rewards_environment.json').write_text(json.dumps(reward_environment, indent=2), encoding='utf-8')
baseline_report = validate_prompt(pipe, PROMPT, SEED, RUN_DIR / 'baseline', repeat=False)
saved = torch.load(RUN_DIR / 'baseline' / 'tensors.pt', map_location='cpu', weights_only=True)
latents_fp16 = saved['custom_latents'].to('cuda')
del saved
print('BASE / Diffusers : PASS | prompt :', PROMPT)
display(Image.open(RUN_DIR / 'baseline' / 'custom.png'))

## 3. Même décodage, avec un graphe autograd

Pour un latent $z$, le décodage est
$y=\mathrm{clamp}(\mathrm{VAE.decode}(z/s)/2+1/2,0,1)$, avec $s=\texttt{vae.config.scaling\_factor}$.
`decode_latents_differentiable` conserve ce graphe ; les paramètres du VAE restent gelés.
On compare les **valeurs** avec `decode_latents` à dtype identique (`atol=rtol=1e-5`).

On place d'abord le U-Net et l'encodeur de texte sur CPU : ils n'interviennent pas dans ce test.
Le checkpointing VAE recalcule des activations pendant le backward, sans modifier l'équation.
Il sera activé pour les tests fp32 sensibles. La génération DDIM précédente reste en fp16.

In [ ]:
pipe.unet.to('cpu')
pipe.text_encoder.to('cpu')
gc.collect()
torch.cuda.empty_cache()

reference_fp16 = decode_latents(pipe, latents_fp16).cpu()
probe = latents_fp16.detach().clone().requires_grad_(True)
differentiable_fp16 = decode_latents_differentiable(pipe.vae, probe)
assert differentiable_fp16.requires_grad and differentiable_fp16.grad_fn is not None
decode_fp16_report = compare_tensors(differentiable_fp16, reference_fp16)
(RUN_DIR / 'decode_fp16.json').write_text(json.dumps(decode_fp16_report, indent=2), encoding='utf-8')
assert decode_fp16_report['finite'] and decode_fp16_report['within_tolerance'], decode_fp16_report
del probe, differentiable_fp16

pipe.vae.to(dtype=torch.float32).eval().requires_grad_(False)
pipe.vae.enable_gradient_checkpointing()
latents = latents_fp16.float().detach()
del latents_fp16
reference_fp32 = decode_latents(pipe, latents).cpu()
probe = latents.detach().clone().requires_grad_(True)
differentiable_fp32 = decode_latents_differentiable(pipe.vae, probe)
decode_fp32_report = compare_tensors(differentiable_fp32, reference_fp32)
(RUN_DIR / 'decode_fp32.json').write_text(json.dumps(decode_fp32_report, indent=2), encoding='utf-8')
assert decode_fp32_report['finite'] and decode_fp32_report['within_tolerance'], decode_fp32_report
assert differentiable_fp32.requires_grad and differentiable_fp32.grad_fn is not None
del probe, differentiable_fp32
image = reference_fp32.to('cuda')
torch.save({'latents': latents.cpu(), 'image_fp32': reference_fp32, 'image_fp16': reference_fp16},
           RUN_DIR / 'reward_inputs.pt')
print('Parité decode fp16 :', decode_fp16_report)
print('Parité decode fp32 avec checkpointing :', decode_fp32_report)
print('Écart fp16/fp32 (diagnostic distinct) :', (reference_fp16 - reference_fp32).abs().max().item())

## 4. Récompenses analytiques

Toutes renvoient `[B]`, avec **plus grand = meilleur**, maximum théorique 0 :

$$\ell(y)=0.299y_R+0.587y_G+0.114y_B,$$
$$R_{lum}=-(\overline{\ell(y)}-\ell^*)^2,\quad
R_{sym}=-\frac{\|y-\mathrm{flip}_{largeur}(y)\|^2}{3HW},\quad
R_{mask}=-\frac{\|\ell(y)-m\|^2}{HW}.$$

Cible provisoire : $\ell^*=0.7$. Masque : disque central de rayon $0.25\min(H,W)$,
valeur 1 à l'intérieur, 0 à l'extérieur. Aucune correction gamma supplémentaire.

Les tests CPU ont comparé autograd aux gradients mathématiques en fp32 :
$-2(\bar\ell-\ell^*)a_c/(HW)$, $-4(y-\mathrm{flip}(y))/(3HW)$ et
$-2(\ell-m)a_c/(HW)$, avec $a=(.299,.587,.114)$.
Le facteur **4** de la symétrie tient compte du gradient des deux occurrences de l'image.

In [ ]:
mask = make_disk_mask(512, 512, radius_fraction=MASK_RADIUS_FRACTION, device='cuda')
reward_functions = {
    'luminance': lambda images: luminance_reward(images, target=TARGET_LUMINANCE),
    'symmetry': symmetry_reward,
    'spatial_mask': lambda images: spatial_mask_reward(images, mask),
}
for name, reward_fn in reward_functions.items():
    print(f'{name:16s} : {reward_fn(image).item(): .8f}')
mask_image = Image.fromarray((mask.cpu().numpy() * 255).astype('uint8'))
mask_image.save(RUN_DIR / 'target_mask.png')
display(mask_image.resize((256, 256)))

## 5. Gradients jusqu'au latent, différences finies et ascension locale

`check_latent_gradient` crée une copie feuille de $z$, calcule $g=\nabla_zR(D(z))$,
puis teste $d=g/\|g\|_2$. On attend
$$\frac{R(D(z+h d))-R(D(z-h d))}{2h}\simeq\|g\|_2$$
et une augmentation locale pour $+h d$. Ce déplacement sert seulement à **tester** le gradient.

Pour les récompenses analytiques, les trois pas $h\in\{0.01,0.003,0.001\}$ sont tous enregistrés.
Le test exige un gradient fini, non nul, sans gradient de paramètres, puis une concordance
et une ascension pour au moins un de ces pas. Tolérance du réseau complet en fp32 :
$10^{-5}+0.10\|g\|_2$. Ce seuil pratique n'est pas celui des tests analytiques, plus stricts.
Les évaluations voisines utilisent aussi autograd avec une entrée feuille, puis libèrent aussitôt
le graphe, pour conserver la même route de calcul que la dérivée.
Le clamp et ses points anguleux peuvent perturber les différences finies ; la fraction de pixels
saturés est enregistrée. Ne pas élargir les seuils pour faire passer un échec.

**ImageReward :** le diagnostic T4 initial a montré que les anciens pas restaient hors du régime
local du score, alors que le VAE linéarisé était cohérent. La grille a donc été affinée après cet échec :
$h\in\{10^{-3},3\cdot10^{-4},10^{-4},3\cdot10^{-5},10^{-5}\}$, avec **deux pas consécutifs**
concordants et donnant une ascension. La tolérance de 10 % est inchangée. Ce choix est désormais fixé
pour les prochains runs ; son origine est décrite dans `validation/`.

**Temps mesuré :** un gradient et six évaluations voisines pour les récompenses analytiques,
dix pour ImageReward, transferts inclus.
Ce coût de diagnostic n'est pas celui d'un seul futur pas de guidance. Les pics incluent les poids résidents.

In [ ]:
gradient_reports = {}
for name, reward_fn in reward_functions.items():
    try:
        (report, gradient), cost = measure_call(check_latent_gradient, pipe.vae, latents, reward_fn)
    except torch.cuda.OutOfMemoryError:
        (RUN_DIR / f'{name}_OOM.txt').write_text(torch.cuda.memory_summary(), encoding='utf-8')
        print('Mémoire GPU insuffisante : diagnostic enregistré. Cette récompense n’est pas validée.')
        raise
    report['cost'] = cost
    gradient_reports[name] = report
    (RUN_DIR / f'{name}_gradient.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    torch.save(gradient.cpu(), RUN_DIR / f'{name}_gradient.pt')
    print(name, json.dumps(report, indent=2))
    del gradient
    assert report['passed'], f'Échec {name} : analyser les dérivées enregistrées.'

## 6. ImageReward officiel : modèle gelé et prétraitement différentiable

`load_imagereward` charge l'architecture officielle, vérifie la révision et le SHA-256 des poids,
puis gèle le réseau en `eval()` et fp32. `image_reward` appelle **`score_gard`** (orthographe officielle),
avec les tokens BERT de longueur 35. Le score contient déjà la standardisation officielle.

Prétraitement : resize bicubique du petit côté à 224 avec antialiasing, crop central 224²,
normalisation aux moyennes/écarts-types CLIP attendus. Le chemin de gradient reste tensoriel.
Le resize s'exécute sur CPU : son backward CUDA ne garantit pas le déterminisme strict.
Les transferts `.to('cpu')` puis `.to('cuda')` conservent autograd et leur coût est mesuré.

**Comparaison à l'API :** `score()` reçoit une image PIL quantifiée en 8 bits. Nous contrôlons d'abord
`score_gard` avec exactement le même tenseur prétraité (`atol=rtol=1e-5`). Ensuite, nous mesurons
l'écart avec notre prétraitement continu de l'image flottante, dont l'interpolation ne reproduit
pas bit à bit PIL. Cet écart est publié séparément, sans promesse d'égalité ou seuil arbitraire.
PIL n'est utilisé que dans ce contrôle de valeurs, jamais sur le chemin du gradient.

Le U-Net et l'encodeur de texte sont déjà sur CPU. ImageReward reste un coût mémoire supplémentaire.
Un OOM est enregistré comme un échec et arrête le notebook ; aucune récompense de substitution n'est utilisée.

In [ ]:
try:
    reward_model, load_cost = measure_call(load_imagereward, device='cuda')
except torch.cuda.OutOfMemoryError:
    (RUN_DIR / 'imagereward_load_OOM.txt').write_text(torch.cuda.memory_summary(), encoding='utf-8')
    print('ImageReward ne tient pas dans ce runtime : étape non validée. Conserver le diagnostic.')
    raise
(RUN_DIR / 'imagereward_load_cost.json').write_text(json.dumps(load_cost, indent=2), encoding='utf-8')
assert not reward_model.training
assert all(not p.requires_grad and p.grad is None for p in reward_model.parameters())
print('ImageReward chargé ; coût de chargement (téléchargement éventuel inclus) :', load_cost)

api_report, api_cost = measure_call(compare_imagereward_api, reward_model, PROMPT, image)
api_report['cost'] = api_cost
(RUN_DIR / 'imagereward_api.json').write_text(json.dumps(api_report, indent=2), encoding='utf-8')
print(json.dumps(api_report, indent=2, ensure_ascii=False))
assert api_report['scores_finite'] and api_report['api_same_input_passed'], api_report

In [ ]:
# Isoler le prétraitement et le réseau avant de leur ajouter le VAE.
leaf_image = image.detach().clone().requires_grad_(True)
score = image_reward(reward_model, PROMPT, leaf_image)
image_gradient = torch.autograd.grad(score.sum(), leaf_image)[0]
image_gradient_report = {
    'score': score.item(), 'shape': list(image_gradient.shape),
    'finite': bool(torch.isfinite(image_gradient).all()), 'gradient_l2': image_gradient.norm().item(),
    'parameter_grads_none': all(p.grad is None for p in reward_model.parameters()),
}
(RUN_DIR / 'imagereward_image_gradient.json').write_text(json.dumps(image_gradient_report, indent=2), encoding='utf-8')
assert score.shape == (1,) and torch.isfinite(score).all()
assert image_gradient.shape == image.shape and image_gradient_report['finite']
assert image_gradient_report['gradient_l2'] > 0 and image_gradient_report['parameter_grads_none']
print('Gradient ImageReward par rapport à l’image :', image_gradient_report)
del leaf_image, score, image_gradient

# Chemin complet : latent → VAE → prétraitement → ImageReward → gradient latent.
try:
    (report, gradient), cost = measure_call(
        check_latent_gradient, pipe.vae, latents,
        lambda images: image_reward(reward_model, PROMPT, images),
        step_sizes=IMAGEREWARD_FD_STEPS, min_consecutive_steps=2,
    )
except torch.cuda.OutOfMemoryError:
    (RUN_DIR / 'imagereward_gradient_OOM.txt').write_text(torch.cuda.memory_summary(), encoding='utf-8')
    print('Backward ImageReward + VAE hors mémoire : étape non validée ; diagnostic sauvegardé.')
    raise
report['cost'] = cost
report['reward_parameters_frozen'] = all(not p.requires_grad for p in reward_model.parameters())
report['reward_parameter_grads_none'] = all(p.grad is None for p in reward_model.parameters())
report['passed'] = report['passed'] and report['reward_parameters_frozen'] and report['reward_parameter_grads_none']
gradient_reports['imagereward'] = report
(RUN_DIR / 'imagereward_gradient.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
torch.save(gradient.cpu(), RUN_DIR / 'imagereward_gradient.pt')
print(json.dumps(report, indent=2))
del gradient
assert report['passed'], 'Gradient ImageReward non validé : lire les différences finies et les normes.'

## 7. Résumé et export

Un succès ici valide les quatre gradients pour **cette image**, ces poids et cette configuration.
Il ne prouve ni un gain de qualité, ni le comportement aux premiers timesteps, ni la mémoire de FULL.
Les scores portent sur le décodage fp32 des mêmes latents BASE ; l'écart avec le décodage fp16
a été mesuré séparément. Aucune optimisation des latents ni génération guidée n'a été effectuée.

Pour reprendre une génération BASE dans cette session, remettre U-Net et encodeur de texte sur CUDA,
désactiver le checkpointing du VAE et remettre le VAE en fp16, puis revalider BASE.
Le plus simple pour la suite est de recharger la pipeline dans une nouvelle session.

In [ ]:
assert set(gradient_reports) == {'luminance', 'symmetry', 'spatial_mask', 'imagereward'}
all_passed = (baseline_report['passed'] and api_report['api_same_input_passed']
              and decode_fp16_report['within_tolerance'] and decode_fp32_report['within_tolerance']
              and all(report['passed'] for report in gradient_reports.values()))
print(f'{"Récompense":16s} {"Score":>12s} {"||gradient||":>14s} {"Test (s)":>10s} {"Pic (Mio)":>12s}')
for name, report in gradient_reports.items():
    print(f'{name:16s} {report["reward"]:12.6f} {report["gradient_l2"]:14.6g} '
          f'{report["cost"]["seconds"]:10.3f} {report["cost"]["peak_allocated_mib"]:12.1f}')
summary = {'passed': all_passed, 'gpu': torch.cuda.get_device_name(0),
           'baseline_passed': baseline_report['passed'], 'api': api_report,
           'decode_fp16': decode_fp16_report, 'decode_fp32': decode_fp32_report,
           'rewards': gradient_reports}
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
assert all_passed, 'Au moins un test reste en échec.'
print('PASS : les quatre gradients sont validés pour cette image. Résultats :', RUN_DIR)

In [ ]:
# Cette cellule peut aussi être exécutée après un échec pour exporter les diagnostics présents.
import shutil
from datetime import datetime, timezone
export_dir = globals().get('RUN_DIR')
if export_dir is None:
    export_dir = PROJECT_DIR / 'runs' / datetime.now(timezone.utc).strftime('setup_%Y%m%d_%H%M%S_UTC')
export_dir.mkdir(parents=True, exist_ok=True)
if (PROJECT_DIR / 'installation_logs').exists():
    shutil.copytree(PROJECT_DIR / 'installation_logs', export_dir / 'installation_logs', dirs_exist_ok=True)
archive_path = shutil.make_archive(str(export_dir), 'zip', root_dir=export_dir)
print('Archive :', archive_path)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(archive_path)